# Harder-arithmetic calibration on Colab

`calibrate_depth.ipynb` showed the single agent solving every chain problem at every depth once it had 8k tokens, with no wrong answers at all: its only failures were running out of tokens at 4k. Voting and decomposition cannot improve on that. This run makes each update harder, `(factor * x + offset) mod 997` with factors up to 99 and offsets up to 199 instead of `mod 97` with factors up to 5, and checks whether the single agent starts making mistakes.

The target is single-agent accuracy roughly between 30% and 70% at 8k tokens, with some `wrong_answer` results and not only `length_limit`. The problems use the development seed with 14 equations per prompt, as in `calibrate_depth`.

Run it as natalia@siwek.poznan.pl, whose Drive holds the `calibrate_depth` results this notebook compares against, and use **Runtime > Change runtime type > L4 GPU** so the precision (bf16) matches the earlier runs. Results go to `MyDrive/small-multi-agents/results/calibrate_hard/`, with the generated problems, a `run.log`, and summary CSV files. If the session disconnects, rerun every cell. Runs resume from the last saved batch.

In [ ]:
RUN_NAME = "calibrate_hard"
SYSTEMS = ["single_agent"]
DEPTHS = [2, 6]
EQUATIONS = 14          # per prompt; the distractor chain fills EQUATIONS - 2 * d
PAIRS_PER_DEPTH = 10
DEV_SEED = 3001         # the main grid uses a different seed
BUDGETS = [8000, 16000]
SAMPLING_SEED = 42
MODULUS, MAX_FACTOR, MAX_OFFSET = 997, 99, 199   # each update is (factor * x + offset) mod MODULUS

assert all(1 <= 2 * d < EQUATIONS for d in DEPTHS), "each prompt needs at least one distractor update"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/small-multi-agents/results/{RUN_NAME}"
!mkdir -p {OUT}

In [ ]:
REPO = "/content/small-multi-agents"
!test -d {REPO} || git clone https://github.com/liatzheng/small-multi-agents.git {REPO}
%cd {REPO}
!git pull --ff-only && git log -1 --format='%H %s'

Hugging Face downloads use your token if it is stored as a Colab secret: open the key icon in the left sidebar, add a secret named `HF_TOKEN`, and turn on notebook access. The token stays in your Colab account, never in the notebook or the repo.

In [ ]:
from notebook_tools import use_hf_token

use_hf_token()

vLLM 0.33.0 brings its own matching torch, torchvision, and transformers, replacing Colab's. The runner runs as a subprocess, so no runtime restart is needed.

In [ ]:
!pip install -q -r requirements-vllm.txt
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__, torch.cuda.get_device_name())" | tee {OUT}/environment.txt
!git rev-parse HEAD | tee -a {OUT}/environment.txt

## Problems

Generation is deterministic, so rerunning this cell rewrites the same file and resuming stays valid.

In [ ]:
PROBLEMS = f"{OUT}/problems.jsonl"
!rm -f {PROBLEMS}
for d in DEPTHS:
    distractor = EQUATIONS - 2 * d
    !python generate_problems.py --pairs {PAIRS_PER_DEPTH} --seed {DEV_SEED} --chain-length {d} --distractor-length {distractor} --modulus {MODULUS} --max-factor {MAX_FACTOR} --max-offset {MAX_OFFSET} >> {PROBLEMS}
!wc -l {PROBLEMS}

## Run

Every system in `SYSTEMS` at every budget, loading the model once. Everything the runner prints is also appended to `run.log`.

In [ ]:
from notebook_tools import run_logged

run_logged(["python", "run_vllm.py", PROBLEMS, "--out-dir", OUT, "--systems", *SYSTEMS,
            "--budgets", *map(str, BUDGETS), "--seed", str(SAMPLING_SEED), "--resume"], f"{OUT}/run.log")

## Results

Accuracy by depth, problem kind, and budget. `length_limit` is the share of problems where the model ran out of tokens before answering, and `wrong_answer` the share where it finished with the wrong pair.

In [ ]:
import pandas as pd
from notebook_tools import load_runs

RESULTS = "/content/drive/MyDrive/small-multi-agents/results"


def summarize(df, by):
    return df.groupby(by).agg(
        accuracy=("correct", "mean"),
        problems=("correct", "size"),
        length_limit=("ran_out", "mean"),
        wrong_answer=("status", lambda s: (s == "wrong_answer").mean()),
        mean_output=("output", "mean"),
    ).round(2)


df = pd.DataFrame(load_runs(OUT))
df.to_csv(f"{OUT}/summary_problems.csv", index=False)
by_depth = summarize(df, ["d", "kind", "budget"])
by_depth.to_csv(f"{OUT}/summary_by_depth.csv")
by_depth

## Compared with `calibrate_depth`

The previous step's saved results, read from Drive, at the same depths and budgets. Both used development seed 3001; only the arithmetic in each update differs.

In [ ]:
previous = pd.DataFrame(load_runs(f"{RESULTS}/calibrate_depth"))
both = pd.concat([previous, df])
both = both[both.d.isin(DEPTHS) & both.budget.isin(BUDGETS)]
compare = summarize(both, ["d", "kind", "budget", "modulus"])
compare.to_csv(f"{OUT}/compare_with_calibrate_depth.csv")
compare["accuracy"].unstack(["budget", "modulus"])

In [ ]:
compare

## Reading the results

- Some `wrong_answer` results and accuracy between roughly 30% and 70% at 8k: these settings give voting and decomposition something to fix. The next step is running the ensemble and hierarchy on them.
- Still near 100% at 8k with no wrong answers: the arithmetic is still too easy. The next step is `mod 9973` with factors up to 999.
- Mostly `length_limit` even at 16k: the harder steps cost too many tokens for these budgets. The next step is fewer equations per prompt or a larger budget.

With 10 pairs per depth, each problem is 10 percentage points within a kind, so treat this as a coarse calibration.